In [2]:
%time
from sentence_transformers import SentenceTransformer
import operator
import regex as re
from pandas import DataFrame
import numpy as np
import umap
#import hdbscan
import matplotlib.pyplot as plt
import pandas as pd
import PyPDF2
import cleantext
from cleantext import clean

---
# Load data

In [3]:
path = "/Users/kazotogbah/Desktop/CapstoneWritting/hmh_data/grade5_corpus_clean_keywords_summarized_distilroberta5d6e638df6bbc73a104a58cba197d9c291bc143038ce348f209bb1ed29ec364b.pkl" 
unpickled_data = pd.read_pickle(path,compression='infer')
unpickled_data

,file,page,text,text_clean,text_clean_no_periods,text_clean_embedded,text_clean_no_periods_embedded,text_keywords,text_keywords_clean_embedded,text_clean_summarized,text_summarized_clean_embedded
0,C:\Users\rdominguez\Documents\Pers\UChicago\Ca...,1,GRADE 5\nTeacher™s \nGuide\nVolume\n1,grade 5 teacher s guide volume 1,grade 5 teacher s guide volume 1,"[0.117546454, -0.1289108, 0.024675569, 0.79411...","[0.117546454, -0.1289108, 0.024675569, 0.79411...",guide volume 1 grade 5 teacher,"[0.17422542, -0.30430984, -0.1412666, 0.402076...",grade 5 teacher s guide volume 1 .,"[0.09989826, -0.12482233, -0.074708804, 0.6445..."
1,C:\Users\rdominguez\Documents\Pers\UChicago\Ca...,2,Authors and Advisors\nAlma Flor Ada Ł Kylene B...,authors and advisors alma flor ada kylene beer...,authors and advisors alma flor ada kylene beer...,"[-0.1704829, 0.5890835, 0.4387237, 0.32914245,...","[-0.14373687, 0.5920101, 0.5037838, 0.35655, -...",probst shane templeton julie washington contri...,"[0.041675113, 0.48674396, 0.4313205, 0.3364311...",teachers and advisors david dockterman mindset...,"[-0.013609975, 0.29976955, 0.21084842, 0.41773..."
2,C:\Users\rdominguez\Documents\Pers\UChicago\Ca...,3,Copyright © 2020 by Houghton Mifflin Harcourt ...,copyright 2020 by houghton mifflin harcourt pu...,copyright 2020 by houghton mifflin harcourt pu...,"[-0.0028390083, 0.063993715, -0.029938823, 0.0...","[-0.023803977, 0.013359526, -0.032527126, 0.07...",isbn 978 0 544 46144 4 1 2 3 4 5 6 7 8 9 10 xx...,"[0.20285466, 0.5847995, 0.4128311, -0.00631434...",copyright 2020 by houghton mifflin harcourt pu...,"[0.2748426, -0.029980134, 0.083095424, -0.1134..."
3,C:\Users\rdominguez\Documents\Pers\UChicago\Ca...,4,CONTENTS\nPROGRAM OVERVIEW\nAuthors and Adviso...,contents program overview authors and advisors...,contents program overview authors and advisors...,"[0.09931215, -0.020393949, 0.19883995, 0.13973...","[0.09931215, -0.020393949, 0.19883995, 0.13973...",advisors ivv develop collaborative self direct...,"[0.046048887, 0.15412024, 0.35991096, 0.457981...",hmh into reading z combines the best practices...,"[0.3145923, -0.03275597, -0.05179701, 0.035828..."
4,C:\Users\rdominguez\Documents\Pers\UChicago\Ca...,5,HMH Into Reading\nŽ\n Authors and Advisors\nCa...,hmh into reading z authors and advisors carol ...,hmh into reading z authors and advisors carol ...,"[0.002761202, 0.20128998, 0.030401845, 0.45410...","[0.038746584, 0.112783305, 0.05895126, 0.50247...",professor emeritus georgia state university na...,"[0.13315715, 0.4420836, -0.06468263, 0.241299,...",nationally known lecturer on reading and writi...,"[0.18033423, -0.10298076, 0.0937723, 0.3509618..."
...,...,...,...,...,...,...,...,...,...,...,...
2563,C:\Users\rdominguez\Documents\Pers\UChicago\Ca...,224,"for specificity, W144\nstrengthen sentences, W...",for specificity w144 strengthen sentences w160...,for specificity w144 strengthen sentences w160...,"[0.07533929, 0.27126107, 0.42523658, 0.7635136...","[0.092685536, 0.27681655, 0.42898053, 0.638846...",specificity w144 strengthen sentences w160 str...,"[0.07472088, 0.34592706, 0.38472912, 0.508255,...",w108 r57 resources do not edit changes must be...,"[0.27958247, 0.11633515, 0.24163221, 0.2382852..."
2564,C:\Users\rdominguez\Documents\Pers\UChicago\Ca...,225,"spelling, frequently misspelled words\n, W343Œ...",spelling frequently misspelled words w343w347 ...,spelling frequently misspelled words w343w347 ...,"[0.160896, 0.42635867, 0.113605425, -0.2727297...","[0.1265479, 0.4114701, 0.20987032, -0.22350144...",early drafts w144 genre preferences w23 groupi...,"[0.358979, 0.31093988, 0.364606, 0.04616851, -...",misspelled words w343w347 connect to writing w...,"[0.047042217, 0.46796146, 0.14991142, -0.38447..."
2565,C:\Users\rdominguez\Documents\Pers\UChicago\Ca...,226,"peer encouragement, W188\npersuasive words and...",peer encouragement w188 persuasive words and p...,peer encouragement w188 persuasive words and p...,"[-0.08801644, 0.30615604, 

In [4]:
path = "/Users/kazotogbah/Desktop/ResearchDesignForBusinessAnalytics/HMH_box/OneCMS_Learning_Spine_English_Language_Arts2021012117.xlsx" 
skills = pd.io.excel.read_excel(path,skiprows=range(0,4),header=1)
skills

,Level 1: Domain,Level 2: Strand,Level 3: Substrand 1,Level 4: Substrand 2,Level 5: Substrand 3,Skill Title,Skill Description,Skill GUID,Skill Code,Lower Grade,Upper Grade
0,Reading and Understanding Text and Media,Comprehending and Analyzing Written Text,General Reading Comprehension Skills,Understanding Author's Purpose and Perspective...,NaN,Identify Intended Audience and Evaluate Its Im...,Identify an informational text's intended audi...,017ab281-a63f-4f41-928e-7d66f29d0bff,CA.GC.1a,K,12
1,Reading and Understanding Text and Media,Comprehending and Analyzing Written Text,General Reading Comprehension Skills,Understanding Author's Purpose and Perspective...,NaN,Identify Author's Purpose While Reading Text,Understand that authors write texts for differ...,5d958ee3-826a-444b-89be-047b54923cd0,CA.GC.1b,K,12
2,Reading and Understanding Text and Media,Comprehending and Analyzing Written Text,General Reading Comprehension Skills,Understanding Author's Purpose and Perspective...,NaN,Analyze How Author Advances Purpose While Reading,"While reading, analyze how the author advances...",efaa706c-b57a-442e-a9ec-bd4a4fd589d1,CA.GC.1c,K,12
3,Reading and Understanding Text and Media,Comprehending and Analyzing Written Text,General Reading Comprehension Skills,Understanding Author's Purpose and Perspective...,NaN,Determine Author's Perspective or Opinions Whi...,Determine the author's perspective and opinion...,015ace30-cf79-4f4e-b973-21bd939a5970,CA.GC.1d,K,12
4,Reading and Understanding Text and Media,Comprehending and Analyzing Written Text,General Reading Comprehension Skills,Understanding Author's Purpose and Perspective...,NaN,Evaluate Author's Credibility to Support Readi...,Identify and analyze an author's qualification...,8d26c54f-a2b6-48ef-9aa6-fad26f0475c5,CA.GC.1e,K,12
...,...,...,...,...,...,...,...,...,...,...,...
2289,"Writing, Researching, Producing, and Presenting",Writing Types and Forms,Writing Poetry,Poetic Writing Types,NaN,Write Haikus or Tankas,"Write haikus or tankas, following the structur...",e0653513-1af9-493c-99cc-ca26440a59cb,WT.WP.2.C4,K,12
2290,"Writing, Researching, Producing, and Presenting",Writing Types and Forms,Writing Poetry,Poetic Writing Types,NaN,Write Riddle Poems,Write riddle poems that require the reader to ...,97410ba5-9fc4-4684-97c4-9be529ceafe5,WT.WP.2.C5,K,12
2291,"Writing, Researching, Producing, and Presenting",Writing Types and Forms,Writing Poetry,Poetic Writing Types,NaN,Write Odes,"Write odes to praise people, events, objects, ...",d5d3e7af-d5b5-45f9-8320-77db880d3856,WT.WP.2.C6,K,12
2292,"Writing, Researching, Producing, and Presenting",Writing Types and Forms,Writing Poetry,Poetic Writing Types,NaN,Write Sonnets,Write sonnets following the structural charact...,7637f149-07f8-4ca0-84d7-5593e1b9211a,WT.WP.2.C7,K,12


In [5]:

path = "/Users/kazotogbah/Desktop/CapstoneWritting/hmh_data/grade5_teachers_guide_pal_corpus_clean_keywords_sumy_distilroberta34b1dc07f9547647fb90760ba6630e5f92fd7af8e4a0d66a84cc084020f76b2d.pkl" 
teachersGuidePal = pd.read_pickle(path,compression='infer')
teachersGuidePal

,file,page,text,content_type,Skill Title,Skill Description,Skill GUID,Skill Code,strand,strand_description,...,Skill Title_embedded,Skill Description_embedded,strand_embedded,Skill Title_no_periods_embedded,Skill Description_no_periods_embedded,strand_no_periods_embedded,strand_description_no_periods_embedded,skill_description_keywords_embedded,strand_description_keywords_embedded,text_clean_summarized_embedded
0,C:\Users\rdominguez\Documents\Pers\UChicago\Ca...,1.0,MODULE\n˜\nﬁ I will not follow \nwhere the pat...,content,module i will not follow where the path may le...,module i will not follow where the path may le...,nan,nan,module i will not follow where the path may le...,module i will not follow where the path may le...,...,"[0.05007931, 0.119613454, 0.032607853, 0.41877...","[0.05007931, 0.119613454, 0.032607853, 0.41877...","[0.05007931, 0.119613454, 0.032607853, 0.41877...","[0.021318758, 0.13580242, 0.051272195, 0.41786...","[0.021318758, 0.13580242, 0.051272195, 0.41786...","[0.021318758, 0.13580242, 0.051272195, 0.41786...","[0.021318758, 0.13580242, 0.051272195, 0.41786...","[0.28464243, 0.2220484, 0.4422115, 0.0465012, ...","[0.28464243, 0.2220484, 0.4422115, 0.0465012, ...","[0.20862822, -0.033599023, 0.044862866, 0.2881..."
1,C:\Users\rdominguez\Documents\Pers\UChicago\Ca...,2.0,MODULE\n˜\nﬁ I will not follow \nwhere the pat...,content,module i will not follow where the path may le...,module i will not follow where the path may le...,nan,nan,module i will not follow where the path may le...,module i will not follow where the path may le...,...,"[0.05007931, 0.119613454, 0.032607853, 0.41877...","[0.05007931, 0.119613454, 0.032607853, 0.41877...","[0.05007931, 0.119613454, 0.032607853, 0.41877...","[0.021318758, 0.13580242, 0.051272195, 0.41786...","[0.021318758, 0.13580242, 0.051272195, 0.41786...","[0.021318758, 0.13580242, 0.051272195, 0.41786...","[0.021318758, 0.13580242, 0.051272195, 0.41786...","[0.30782643, 0.058543533, 0.16792017, 0.085862...","[0.30782643, 0.058543533, 0.16792017, 0.085862...","[0.16788949, -0.13963626, -0.07024216, 0.53562..."
2,C:\Users\rdominguez\Documents\Pers\UChicago\Ca...,3.0,˜˚˛˝˙ˆˇ˘˝\n\nˆˇ\nThe words in the chart will h...,content,the words in the chart will help you talk and ...,the words in the chart will help you talk and ...,nan,nan,the words in the chart will help you talk and ...,the words in the chart will help you talk and ...,...,"[0.24913393, -0.16405915, -0.045558963, -0.163...","[0.24913393, -0.16405915, -0.045558963, -0.163...","[0.24913393, -0.16405915, -0.045558963, -0.163...","[0.2626564, -0.15431736, -0.14201987, -0.11405...","[0.2626564, -0.15431736, -0.14201987, -0.11405...","[0.2626564, -0.15431736, -0.14201987, -0.11405...","[0.2626564, -0.15431736, -0.14201987, -0.11405...","[0.318849, 0.5429922, 0.2398742, -0.08303005, ...","[0.318849, 0.5429922, 0.2398742, -0.08303005, ...","[0.23586863, 0.037411276, 0.0036898185, 0.0774..."
3,C:\Users\rdominguez\Documents\Pers\UChicago\Ca...,4.0,˜˚˛˝˙ˆˇ˘˝\n\nˆˇ\nThe words in the chart will h...,content,the words in the chart will help you talk and ...,the words in the chart will help you talk and ...,nan,nan,the words in the chart will help you talk and ...,the words in the chart will help you talk and ...,...,"[0.24913393, -0.16405915, -0.045558963, -0.163...","[0.24913393, -0.16405915, -0.045558963, -0.163...","[0.24913393, -0.16405915, -0.045558963, -0.163...","[0.2626564, -0.15431736, -0.14201987, -0.11405...","[0.2626564, -0.15431736, -0.14201987, -0.11405...","[0.2626564, -0.15431736, -0.14201987, -0.11405...","[0.2626564, -0.15431736, -0.14201987, -0.11405...","[0.34651068, 0.33815506, 0.31886512, -0.218285...","[0.34651068, 0.33815506, 0.31886512, -0.218285...","[0.13544446, 0.15687963, -0.033667527, -0.0243..."
4,C:\Users\rdominguez\Documents\Pers\UChicago\Ca...,5.0,˜\n˚˛˝˙˜\n\nReasons to \nInvent\nSolve \nProbl...,content,reasons to invent solve problems achieve fame ...,reasons to invent solve problems achi

In [15]:
teachersGuidePal['text'][0]

"MODULE\n˜\nﬁ I will not follow \nwhere the path may \nlead, but I will go \nwhere there is no \npath, and I will leave \na trail.ﬂ\nŠMuriel Strode\nInventors \nat Work\nInventors \nat Work\n10\nDO NOT EDIT--Changes must be made through ﬁFile infoﬂ\nCorrectionKey=TX-A\nDO NOT EDIT--Changes must be made through ﬁFile infoﬂ\nCorrectionKey=TX-A\n5re_se_m1_mo.indd   10\n2/26/2018   5:35:52 AM\n˜˚˛˛˝˙ˆˇ˘˜˝˛ˆˇ˙\nWhat kinds of \ncircumstances \npush˜people \nto create new \ninventions?\n\n11\nDO NOT EDIT--Changes must be made through ﬁFile infoﬂ\nCorrectionKey=TX-B;NL-B\n5re_se_m1_mo.indd   11\n11/2/2018   12:53:07 AM\nIntroduce the Topic\nŁ\n \nRead aloud \nthe module title, \nInventors at Work.\nŁ\n \nTell students\n that in this module \nthey will be reading and viewing \nselections on the topic of inventors.\nŁ\n \nHave students\n brainstorm famous \ninventors and share what they know \nabout their inventions.\nŁ\n \nThen ask students to discuss the \nmodule title and explain the \ndiffer

In [19]:
teachersGuidePal['text'][500]

'˜˚˛˝˙ˆˇˆ˘ˇ˙˝˙˛˝\nP\nROMPT\nThe Mighty Mars Rovers\n describes what happened when the rover vehicles Spirit and \nOpportunity landed on and explored the surface of Mars.\nImagine that you train new rover drivers at NASA™s Jet Propulsion Laboratory. You need \nto be sure the new drivers understand how to control the rovers and don™t make \nmistakes that could cause damage. Use evidence from the text to prepare a safety \nchecklist that new drivers can use to avoid damaging the rovers. Use precise, clear \nlanguage. Don™t forget to use some of the Critical Vocabulary words in your writing.\nPLAN\nMake notes from the text about things a rover driver should do to avoid \ndamaging the rover. \n˛˝˙˝ˆ\n˙\n140\nDO NOT EDIT--Changes must be made through ﬁFile infoﬂ\nCorrectionKey=TX-A\nDO NOT EDIT--Changes must be made through ﬁFile infoﬂ\nCorrectionKey=TX-A\n5re_se_m7_marsrover_pr.indd   140\n2/26/2018   6:22:42 AM\nWrite About Reading\nŁ\n \nRead aloud \nthe prompt with \nstudents.\nŁ\n \nLea

In [16]:
teachersGuidePal['Skill Description'][5]

'reasons to invent solve problems achieve fame and fortune 14 do not edit changes must be made through file infofl correctionkey=tx a do not edit changes must be made through file infofl correctionkey=tx a 5re_se_m1_km.indd 14 2/26/2018 5 36 29 am make life easier entertain people 15 do not edit changes must be made through file infofl correctionkey=tx a 5re_se_m1_km.indd 15 2/26/2018 5 36 30 am inventors at work 15 do not edit changes must be made through file infofl correctionkey=tx a'

In [17]:
teachersGuidePal['content_type'][0]

'content'

In [24]:
import pandas as pd
import numpy as np
pal_skills = []
pal_content = []
page = -1
for text in teachersGuidePal['content_type']:
    page = page+1
    if teachersGuidePal['content_type'][page]== "content":
        pal_content.append(teachersGuidePal['Skill Description'][page])
    else:
        pal_skills.append(teachersGuidePal['Skill Description'][page])


In [26]:
pal_content[0]

"module i will not follow where the path may lead but i will go where there is no path and i will leave a trail.fl smuriel strode inventors at work inventors at work 10 do not edit changes must be made through file infofl correctionkey=tx a do not edit changes must be made through file infofl correctionkey=tx a 5re_se_m1_mo.indd 10 2/26/2018 5 35 52 am what kinds of circumstances push people to create new inventions? 11 do not edit changes must be made through file infofl correctionkey=tx b nl b 5re_se_m1_mo.indd 11 11/2/2018 12 53 07 am introduce the topic read aloud the module title inventors at work. tell students that in this module they will be reading and viewing selections on the topic of inventors. have students brainstorm famous inventors and share what they know about their inventions. then ask students to discuss the module title and explain the different types of work inventors may do to come up with an invention that works. (possible responses inventors do a lot of researc

In [28]:
len(pal_content)

3288

In [27]:
pal_skills[0]

"identify an informational text's intended audience while reading and analyze how it affects the author's development of a text"

In [29]:
len(pal_skills)

2294

##### ---
# Q&A Model

In [7]:
from transformers import BertForQuestionAnswering
#from transformers import AutoModel

In [8]:
model = BertForQuestionAnswering.from_pretrained('deepset/bert-base-cased-squad2')
#model = BertForQuestionAnswering.from_pretrained('deepset/electra-cased-squad2')

In [9]:
from transformers import AutoTokenizer
#from transformers import BertTokenizer

In [10]:
tokenizer = AutoTokenizer.from_pretrained('deepset/bert-base-cased-squad2')
#tokenizer.encode(skills["Skill Description"][0],truncation=True,padding=True)

In [11]:
from transformers import pipeline

In [12]:
bert_qa = pipeline('question-answering',model =model,tokenizer=tokenizer)

In [ ]:
bert_qa({'context':unpickled_data['text_clean'][39] ,'question': skills["Skill Description"][0]})

____
## skills_to_questions

In [30]:
import pandas as pd
import numpy as np
skills_to_questions = []
skillnum = -1
concat_1 = "How to "
concat_2 = "?"
for skill in skills["Skill Description"]:
    skillnum = skillnum+1
    skills_to_questions.append(concat_1 + skills["Skill Description"][skillnum] + concat_2)
    
skills_to_questions  

["How to Identify an informational text's intended audience while reading, and analyze how it affects the author's development of a text?",
 "How to Understand that authors write texts for different purposes: to inform, persuade, entertain, or share thoughts or feelings; and identify author's purpose(s) while reading text?",
 'How to While reading, analyze how the author advances the purpose of a text in terms of style and use of text structure, language and rhetoric (e.g., formal or informal language, figures of speech, tone, persuasiveness), and evaluate the degree to which the author achieved the purpose of the text?',
 "How to Determine the author's perspective and opinions, including underlying values and beliefs, based on explicit and implicit information while reading text?",
 "How to Identify and analyze an author's qualifications to evaluate the author's credibility?",
 "How to While reading, evaluate how an author's personal, cultural, and historical background impacts their 

In [32]:
import pandas as pd
import numpy as np
skills_to_Quest = []
skillnum = -1
concat_1 = "How to "
concat_2 = "?"
for skill in pal_skills:
    skillnum = skillnum+1
    skills_to_Quest.append(concat_1 + pal_skills[skillnum] + concat_2)
    
skills_to_Quest  


["How to identify an informational text's intended audience while reading and analyze how it affects the author's development of a text?",
 "How to understand that authors write texts for different purposes to inform persuade entertain or share thoughts or feelings and identify author's purpose(s) while reading text?",
 'How to while reading analyze how the author advances the purpose of a text in terms of style and use of text structure language and rhetoric (e.g. formal or informal language figures of speech tone persuasiveness) and evaluate the degree to which the author achieved the purpose of the text?',
 "How to determine the author's perspective and opinions including underlying values and beliefs based on explicit and implicit information while reading text?",
 "How to identify and analyze an author's qualifications to evaluate the author's credibility?",
 "How to while reading evaluate how an author's personal cultural and historical background impacts their purpose for writin

In [13]:
skills_to_question[0]

"How to Identify an informational text's intended audience while reading, and analyze how it affects the author's development of a text?"

In [17]:
bert_qa({'context':unpickled_data['text_clean'][39] ,'question': skills["Skill Description"][0]})

{'score': 0.06018202006816864,
 'start': 237,
 'end': 333,
 'answer': 'author s purpose central ideas and text structure in order to better understand unfamiliar texts'}

In [18]:
bert_qa({'context':unpickled_data['text_clean'][40] ,'question': skills["Skill Description"][0]})

{'score': 0.02865724079310894,
 'start': 621,
 'end': 648,
 'answer': 'building knowledge networks'}

In [19]:
bert_qa({'context':unpickled_data['text_clean'][39] ,'question': skills_to_questions[0]})

{'score': 0.12295632809400558,
 'start': 152,
 'end': 187,
 'answer': 'a genre focus on informational text'}

In [21]:
bert_qa({'context':unpickled_data['text_clean'][40] ,'question': skills_to_questions[0]})

{'score': 0.4144015908241272,
 'start': 621,
 'end': 648,
 'answer': 'building knowledge networks'}

In [24]:
bert_qa({'context':unpickled_data['text_clean'][41] ,'question': skills_to_questions[1]})

{'score': 2.5065095542231575e-05,
 'start': 293,
 'end': 315,
 'answer': 'key messages try smart'}

In [27]:
bert_qa({'context':unpickled_data['text_clean'][39] ,'question': skills_to_questions[1]})

{'score': 0.1698533296585083,
 'start': 152,
 'end': 187,
 'answer': 'a genre focus on informational text'}

In [28]:
bert_qa({'context':unpickled_data['text_clean'][38] ,'question': skills_to_questions[1]})

{'score': 1.3364928008741117e-06,
 'start': 71,
 'end': 91,
 'answer': 'carefully controlled'}

___
## Score outputs

In [33]:
import pandas as pd
import numpy as np
page = 0
skillnum = 0
qa_scores = []
for page in range(0,10):
    page =page+1
    context = pal_content[page]
    score=[]
    for skillnum in range(0,10):
        question = skills_to_Quest[skillnum]
        skillnum =skillnum+1
        score.append(bert_qa({'context':context,'question':question})['score'])
    qa_scores.append(score)
        
        
score_df_clean_text = pd.DataFrame(qa_scores)

In [34]:
score_df_clean_text

,0,1,2,3,4,5,6,7,8,9
0,0.000347,0.000006,6.240523e-04,0.000698,0.000012,0.001284,0.000140,0.000066,0.004953,0.013247
1,0.011541,0.000021,1.675111e-02,0.002140,0.000051,0.015167,0.002206,0.000304,0.003653,0.014672
2,0.011491,0.000387,1.387742e-02,0.003464,0.000179,0.014533,0.003572,0.000155,0.003038,0.010954
3,0.005909,0.000218,8.847508e-05,0.000194,0.000004,0.000035,0.000601,0.000097,0.000258,0.009371
4,0.000184,0.000056,3.962576e-07,0.000262,0.000003,0.000003,0.000474,0.000007,0.000033,0.000041
5,0.006358,0.005223,2.068666e-02,0.000419,0.000023,0.006030,0.003546,0.000006,0.051151,0.000183
6,0.173539,0.035234,1.270203e-06,0.070302,0.069619,0.064156,0.047348,0.144121,0.024672,0.018258
7,0.009970,0.002568,6.152774e-02,0.000564,0.000692,0.001290,0.000030,0.000004,0.010884,0.003322
8,0.014746,0.007951,4.768683e-02,0.013628,0.000031,0.135329,0.001272,0.008344,0.016226,0.027081
9,0.011323,0.002462,2.993201e-02,0.025517,0.000004,0.004563,0.000685,0.000531,0.017471,0.000881


In [32]:
import pandas as pd
import numpy as np
page = 40
skillnum = 0
qa_scores = []
for page in range(40,140):
    page =page+1
    context = unpickled_data['text_clean'][page]
    score=[]
    for skillnum in range(100):
        # just added "how to" and question mark (?) to evrey skill
        question = skills_to_questions[skillnum]
        skillnum =skillnum+1
        score.append(bert_qa({'context':context,'question':question})['score'])
    qa_scores.append(score)
        
        
score_df_clean_text = pd.DataFrame(qa_scores)

In [33]:
score_df_clean_text

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,2.506510e-05,0.000548,7.575395e-07,0.000002,0.000116,0.000088,4.967525e-06,0.000188,0.002121,0.000003,...,0.142779,0.000083,0.001284,0.162543,0.000350,0.000311,0.001555,0.000201,0.000037,0.000072
1,1.054764e-02,0.012798,4.200072e-02,0.107252,0.009440,0.002344,1.283436e-03,0.045278,0.039394,0.006943,...,0.003251,0.000330,0.000159,0.026373,0.001811,0.000510,0.015580,0.000085,0.000379,0.025353
2,1.301828e-05,0.087920,8.295783e-04,0.000025,0.121273,0.000118,7.147056e-05,0.086332,0.010634,0.000738,...,0.010423,0.000162,0.003894,0.059479,0.000206,0.005838,0.021012,0.000068,0.000099,0.000163
3,9.267115e-07,0.001294,1.010662e-04,0.000068,0.005173,0.000051,8.446116e-07,0.008732,0.000824,0.000004,...,0.003816,0.000025,0.003013,0.052475,0.000345,0.015101,0.001741,0.000026,0.000026,0.000014
4,4.147178e-05,0.000336,5.394928e-04,0.000023,0.000613,0.001698,4.254079e-05,0.005226,0.000246,0.002861,...,0.000492,0.000042,0.000100,0.007181,0.000055,0.000571,0.002260,0.000086,0.000176,0.000022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,7.823324e-07,0.000078,1.053043e-06,0.000022,0.002149,0.002341,1.031484e-06,0.008306,0.002946,0.012144,...,0.000233,0.000144,0.000269,0.108006,0.002706,0.000565,0.002308,0.000093,0.000080,0.000070
96,2.533806e-05,0.003429,4.900563e-04,0.024559,0.017088,0.019399,5.016596e-04,0.527397,0.011263,0.018875,...,0.018588,0.002409,0.024329,0.059540,0.004958,0.113895,0.024212,0.026556,0.001228,0.012906
97,7.994079e-04,0.019746,6.129724e-05,0.000009,0.031758,0.000007,1.063395e-06,0.006522,0.001520,0.000219,...,0.003134,0.000587,0.008881,0.326911,0.000719,0.007693,0.002110,0.008861,0.000255,0.000857
98,8.950134e-05,0.000750,9.375910e-06,0.000085,0.010134,0.000291,5.655368e-05,0.031551,0.001010,0.000244,...,0.000285,0.036143,0.000892,0.030420,0.000900,0.001618,0.002643,0.007989,0.001175,0.012734


In [49]:
bert_qa({'context':unpickled_data['text_clean'][137],'question':skills_to_questions[8]})['score']

0.5273969173431396

In [50]:
bert_qa({'context':unpickled_data['text_clean'][137],'question':skills_to_questions[8]})

{'score': 0.5273969173431396,
 'start': 2464,
 'end': 2492,
 'answer': 'reread the sentence fluently'}

In [51]:
unpickled_data['text_clean'][137]

'foundational skills fluency english learner support support comprehension all levels as you model reading accurately have students raise their hands when they hear a word they do not recognize. work with students to decode the word and practice saying it aloud. discuss the word s meaning using gestures or pictures for support if needed. then have students read the entire sentence chorally. provide corrective feedback as needed. high frequency words point out the high frequency words in the passage on printable fluency 1.6 . remind students that high frequency words appear often in texts they read. students can learn to recognize them rather than decode them so that they can read more fluently. print and distribute word cards 5.55.8 which feature this week s high frequency words and have students work independently or in pairs to read and complete the activities for each word. for struggling readers walk through the notes for one or two words before they continue working with a partner

In [54]:
skills_to_questions[8]

"How to Evaluate how an author's perspective or opinions impact a text while reading?"

In [45]:
bert_qa({'context':unpickled_data['text_clean'][140],'question':skills_to_questions[8]})['score']

0.36513105034828186

In [46]:
bert_qa({'context':unpickled_data['text_clean'][140],'question':skills_to_questions[8]})

{'score': 0.36513105034828186,
 'start': 87,
 'end': 101,
 'answer': 'text structure'}

In [48]:
skills_to_questions[8]

"How to Evaluate how an author's perspective or opinions impact a text while reading?"

In [47]:
unpickled_data['text_clean'][140]

'minilesson shared reading reading workshop step 1 connect and teach tell students that text structure is the way an author organizes information within a text. a text s structure can order events or steps in a sequence show contrasts between two things or help readers see the relationships between ideas and events. project or display anchor chart 18 text structure. point out to students that without text structure it would be difficult for readers to follow complicated ideas or to understand the order in which things happen. point out also that every event has a cause and an effect. the cause of an event is the why or the reason something happened. the effect is the what or the result of what happened. tell students that an author uses certain words to signal each kind of text structure. sequence words such as before next later and finally show the order in which things happen for example while words such as so because if and then show cause and effect relationships. tell students tha

In [52]:
import pandas as pd
import numpy as np
page = 40
skillnum = 0
qa_answers = []
for page in range(40,45) :
    page = page+1
    context = unpickled_data['text_clean'][page]
    answer=[]
    for skillnum in range(0,5):
        skillnum =skillnum+1
        question = skills_to_questions[skillnum]
        answer.append(bert_qa({'context':context,'question':question})['answer'])
    qa_answers.append(answer)
        
        
answer_df_clean_text = pd.DataFrame(qa_answers)

In [53]:
answer_df_clean_text

,0,1,2,3,4
0,key messages try smart,focuss trying again,sand use the suggestions to weave it throughou...,display anchor chart 32,key messages try smart
1,author s purpose,craft,generate a plan speaking and listening,retell/summarize monitor,author s craft response to text write
2,writing,reading,writing process informational text plan and ge...,reading rate accuracy and self correction,reading rate accuracy and self correction
3,carefully selected content rich text sets help...,video teaching,students,video teaching with text sets carefully select...,guided reading level s read aloud get curious ...
4,genre informational text lexile,guided reading,fifile,fifile,guided reading level v writing focal text genr...


In [ ]:
import pandas as pd
import numpy as np
qa_scores = []
page = -1
skillnum = -1
for text in unpickled_data['text_clean'] :
    page =+1
    context = unpickled_data['text_clean'][page]
    score=[]
    for skill in skills_to_questions:
        skillnum = skillnum+1
        question = skills_to_questions[skillnum]
        score.append(bert_qa({'context':context,'question':question})['score'])
    qa_scores.append(score)
        
        
score_df_clean_text = pd.DataFrame(qa_scores)




"""This Block throw in the list out of range error bellow
    and I don't see how to fix it. """ 
#IndexError                                
#IndexError: list index out of range


____
## skills_to_questions_2

In [58]:
import pandas as pd
import numpy as np
skills_to_questions2 = []
skillnum = -1
concat_1 = "what about "
concat_2 = "?"
for skill in skills["Skill Description"]:
    skillnum = skillnum+1
    skills_to_questions2.append(concat_1 + skills["Skill Description"][skillnum] + concat_2)
    
skills_to_questions2  

["what about Identify an informational text's intended audience while reading, and analyze how it affects the author's development of a text?",
 "what about Understand that authors write texts for different purposes: to inform, persuade, entertain, or share thoughts or feelings; and identify author's purpose(s) while reading text?",
 'what about While reading, analyze how the author advances the purpose of a text in terms of style and use of text structure, language and rhetoric (e.g., formal or informal language, figures of speech, tone, persuasiveness), and evaluate the degree to which the author achieved the purpose of the text?',
 "what about Determine the author's perspective and opinions, including underlying values and beliefs, based on explicit and implicit information while reading text?",
 "what about Identify and analyze an author's qualifications to evaluate the author's credibility?",
 "what about While reading, evaluate how an author's personal, cultural, and historical b

In [59]:
import pandas as pd
import numpy as np
page = 40
skillnum = 0
qa_answers2 = []
for page in range(40,45) :
    page = page+1
    context = unpickled_data['text_clean'][page]
    answer=[]
    for skillnum in range(0,5):
        skillnum =skillnum+1
        question = skills_to_questions2[skillnum]
        answer.append(bert_qa({'context':context,'question':question})['answer'])
    qa_answers2.append(answer)
        
        
answer_df_clean_text2 = pd.DataFrame(qa_answers2)

In [60]:
answer_df_clean_text2

,0,1,2,3,4
0,key messages try smart,learning mindset feature to introduce the lear...,key messages try smart,it s okay (and expected) to make mistakes when...,key messages try smart
1,literary elements/author s purpose and craft a...,craft,generate a plan speaking and listening,make inferences,literary elements
2,writing process informational text plan and ge...,reading rate accuracy and self correction,reading rate accuracy and self correction,reading rate accuracy and self correction,reading rate accuracy and self correction phra...
3,carefully selected content rich text sets help...,carefully selected content rich text sets help...,genre informational text my book my book morni...,genre opinion essay lexile,carefully selected content rich text sets help...
4,genre informational text lexile,what kinds of circumstances push people to cre...,genre informational text lexile,inventors at work genre science fiction/fantas...,genre informational text lexile


In [62]:
%time
import pandas as pd
import numpy as np
page = 40
skillnum = 0
qa_scores2 = []
for page in range(40,45):
    context = unpickled_data['text_clean'][page]
    page =page+1
    score=[]
    for skillnum in range(0,4):
        question = skills_to_questions2[skillnum]
        skillnum =skillnum+1
        score.append(bert_qa({'context':context,'question':question})['score'])
    qa_scores2.append(score)
        
        
score_df_clean_text2 = pd.DataFrame(qa_scores2)

In [63]:
score_df_clean_text2

,0,1,2,3
0,0.092559,0.001113,0.002930,0.006580
1,0.000447,0.001456,0.000029,0.000001
2,0.000340,0.030614,0.013645,0.012731
3,0.020783,0.000110,0.071343,0.000667
4,0.000618,0.000026,0.000265,0.000584


In [35]:
import pandas as pd